In [1]:
import ast
import glob
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
all_data = []

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    replicon_data = pd.read_csv(f'{folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_data.rename(columns={'accession': 'contig'}, inplace=True)
    
    data_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/annotations'
    mob_data = pd.read_csv(f'{data_dir}/NMS_replicon_mobtyper_results.tsv', sep = '\t', index_col=0)
    mob_data.rename(columns={'id': 'contig'}, inplace=True)
    amr_data = pd.read_csv(f'{data_dir}/NMS_replicon_AMRtyper_results.tsv', sep = '\t', index_col=0)
    amr_data = amr_data.drop_duplicates(subset=['accession'], keep='first')
    amr_data.rename(columns={'accession': 'contig'}, inplace=True)
    merged_data = pd.merge(mob_data, amr_data, on='contig', how='left')
    merged_data['genus'] = genus_name

    merged_data = pd.merge(merged_data, replicon_data[['contig', 'category-pident_90']], on='contig', how='left')
    all_data.append(merged_data)

all_data = pd.concat(all_data, ignore_index=True)

In [3]:
all_data = all_data[['contig', 'genus', 'size', 'category-pident_90', 'rep_type(s)', 'relaxase_type(s)', 'mpf_type',
       'orit_type(s)', 'predicted_mobility', 'AMR_type']]
all_data = all_data[all_data['category-pident_90'].isin(['typical plasmid', 'intermediate replicon'])]

target_dir = '/active-data/analysis_results/chr_pla/genus/suptables'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
all_data.to_csv('replicon_mobility_and_AMR_profile_data.tsv', sep='\t', index=False)